# Heat Exchanger Research Notebook

## End-to-End, Physics-Aware, From-Scratch Regression Project

This notebook is designed for Google Colab and written like a research workbook.

We will:
1. Upload and inspect the dataset
2. Clean and validate the data
3. Engineer physics-aware input features
4. Separate training-safe features from diagnostic features
5. Build multiple regression models from scratch
6. Compare models using tables and plots
7. Run experiments for two important targets:
   - `hx_1_heat_load_kw` as the primary target
   - `hot_outlet_temperature_k` as the secondary target
8. Export predictions, metrics, and serialized artifacts

## Important Modeling Philosophy

This notebook is built for research and extensibility:
- It does not use outlet-side columns as model inputs
- It introduces fluid-property placeholders such as specific heat, density, viscosity, and thermal conductivity
- It computes physically meaningful derived quantities such as inlet temperature difference, heat capacity rates, capacity-rate ratio, thermal diffusivity, Prandtl number, and energy driving-force proxies


## How To Use This Notebook In Google Colab

1. Open this notebook in Google Colab
2. Run the import/setup cell
3. Upload `heat_exchanger_dataset.csv` when prompted
4. Run the notebook from top to bottom
5. Review the analysis, feature engineering, models, and plots
6. Download the exported CSV and artifact files at the end

## Notes About This Dataset

Some columns appear constant or physically unusual. This notebook handles that in a research-friendly way:
- constant columns are identified explicitly
- they are not blindly trusted as useful predictors
- target-side quantities are used only for post-hoc diagnostics, not as training inputs


In [ ]:
import math
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from google.colab import files
except ImportError:
    files = None

warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)


## Step 1: Upload The Dataset

This cell is Colab-friendly. When running in Google Colab, it will prompt you to upload the CSV file.


In [ ]:
DATA_FILE = Path("heat_exchanger_dataset.csv")

if files is not None:
    print("Please upload heat_exchanger_dataset.csv")
    uploaded = files.upload()
    if DATA_FILE.name not in uploaded:
        print("Uploaded file name does not exactly match heat_exchanger_dataset.csv")
else:
    print("Google Colab upload helper is not available here. Make sure the CSV is already in the working directory.")

if not DATA_FILE.exists():
    raise FileNotFoundError("heat_exchanger_dataset.csv not found. Upload it before continuing.")

print(f"Dataset found at: {DATA_FILE.resolve()}")


## Step 2: Project Configuration And Physical Assumptions

Because the dataset does not explicitly provide every inlet-side fluid property, we define a configurable block here.

### Assumptions used in this notebook
- Cold inlet temperature is assumed constant at 293.15 K (20°C)
- Hot-side mass flow is assumed constant at 1.0 kg/s
- Default fluid properties are set to water-like values

These are deliberately centralized so you can later swap in other fluids.


In [ ]:
CONFIG = {
    "random_state": 42,
    "test_size": 0.20,
    "validation_fraction_of_train": 0.20,
    "cold_inlet_temperature_k": 293.15,
    "assumed_hot_mass_flow_kg_s": 1.0,
    "cp_hot_kj_kgk": 4.18,
    "cp_cold_kj_kgk": 4.18,
    "rho_hot_kg_m3": 997.0,
    "rho_cold_kg_m3": 997.0,
    "mu_hot_pa_s": 0.0010,
    "mu_cold_pa_s": 0.0010,
    "k_hot_w_mk": 0.60,
    "k_cold_w_mk": 0.60,
}

display(pd.DataFrame({"Parameter": list(CONFIG.keys()), "Value": list(CONFIG.values())}))


## Step 3: Load And Inspect The Raw Dataset

Before building any model, we inspect shape, dtypes, missing values, duplicate rows, summary statistics, and constant columns.


In [ ]:
df_raw = pd.read_csv(DATA_FILE)

print("Dataset shape:", df_raw.shape)
print("Columns:")
print(df_raw.columns.tolist())

display(df_raw.head())
display(df_raw.describe(include="all").T)


In [ ]:
print("Data types:")
display(df_raw.dtypes.to_frame("dtype"))

print("Missing values per column:")
display(df_raw.isnull().sum().to_frame("missing_count"))

print("Duplicate rows:", int(df_raw.duplicated().sum()))

display(pd.DataFrame({
    "column": df_raw.columns,
    "nunique": [df_raw[col].nunique() for col in df_raw.columns]
}).sort_values("nunique").head(10))


## Step 4: Data Cleaning

This section removes exact duplicates, enforces numeric columns, and drops missing rows if needed.


In [ ]:
df = df_raw.copy().drop_duplicates().reset_index(drop=True)
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_after_numeric = int(df.isnull().sum().sum())
print("Total missing values after numeric coercion:", missing_after_numeric)
if missing_after_numeric > 0:
    df = df.dropna().reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)


## Step 5: Physics-Aware Feature Engineering

We now create inlet-side features and physics-inspired derived features.
These use only inlet values, noisy inlet values, and assumed constants.


In [ ]:
def build_feature_frame(df_in, config):
    out = pd.DataFrame(index=df_in.index)
    cold_inlet_temperature_k = config["cold_inlet_temperature_k"]
    hot_mass_flow_kg_s = config["assumed_hot_mass_flow_kg_s"]

    cp_hot_kj_kgk = config["cp_hot_kj_kgk"]
    cp_cold_kj_kgk = config["cp_cold_kj_kgk"]
    cp_hot_j_kgk = cp_hot_kj_kgk * 1000.0
    cp_cold_j_kgk = cp_cold_kj_kgk * 1000.0

    rho_hot = config["rho_hot_kg_m3"]
    rho_cold = config["rho_cold_kg_m3"]
    mu_hot = config["mu_hot_pa_s"]
    mu_cold = config["mu_cold_pa_s"]
    k_hot = config["k_hot_w_mk"]
    k_cold = config["k_cold_w_mk"]

    out["hot_inlet_temperature_k"] = df_in["hot_inlet_temperature_k"]
    out["hot_inlet_temperature_k_noisy"] = df_in["hot_inlet_temperature_k_noisy"]
    out["cold_inlet_mass_flow_kg_s"] = df_in["cold_inlet_mass_flow_kg_s"]
    out["cold_inlet_mass_flow_kg_s_noisy"] = df_in["cold_inlet_mass_flow_kg_s_noisy"]
    out["cold_inlet_temperature_k_assumed"] = cold_inlet_temperature_k
    out["hot_mass_flow_kg_s_assumed"] = hot_mass_flow_kg_s

    out["hot_inlet_sensor_error_k"] = out["hot_inlet_temperature_k_noisy"] - out["hot_inlet_temperature_k"]
    out["cold_flow_sensor_error_kg_s"] = out["cold_inlet_mass_flow_kg_s_noisy"] - out["cold_inlet_mass_flow_kg_s"]

    out["delta_t_in_clean_k"] = out["hot_inlet_temperature_k"] - cold_inlet_temperature_k
    out["delta_t_in_noisy_k"] = out["hot_inlet_temperature_k_noisy"] - cold_inlet_temperature_k

    out["cp_hot_kj_kgk"] = cp_hot_kj_kgk
    out["cp_cold_kj_kgk"] = cp_cold_kj_kgk
    out["rho_hot_kg_m3"] = rho_hot
    out["rho_cold_kg_m3"] = rho_cold
    out["mu_hot_pa_s"] = mu_hot
    out["mu_cold_pa_s"] = mu_cold
    out["k_hot_w_mk"] = k_hot
    out["k_cold_w_mk"] = k_cold

    out["capacity_rate_hot_kw_per_k"] = hot_mass_flow_kg_s * cp_hot_kj_kgk
    out["capacity_rate_cold_clean_kw_per_k"] = out["cold_inlet_mass_flow_kg_s"] * cp_cold_kj_kgk
    out["capacity_rate_cold_noisy_kw_per_k"] = out["cold_inlet_mass_flow_kg_s_noisy"] * cp_cold_kj_kgk

    out["capacity_rate_ratio_clean"] = out["capacity_rate_cold_clean_kw_per_k"] / out["capacity_rate_hot_kw_per_k"]
    out["capacity_rate_ratio_noisy"] = out["capacity_rate_cold_noisy_kw_per_k"] / out["capacity_rate_hot_kw_per_k"]

    out["capacity_rate_min_clean"] = np.minimum(out["capacity_rate_hot_kw_per_k"], out["capacity_rate_cold_clean_kw_per_k"])
    out["capacity_rate_max_clean"] = np.maximum(out["capacity_rate_hot_kw_per_k"], out["capacity_rate_cold_clean_kw_per_k"])
    out["capacity_rate_min_noisy"] = np.minimum(out["capacity_rate_hot_kw_per_k"], out["capacity_rate_cold_noisy_kw_per_k"])
    out["capacity_rate_max_noisy"] = np.maximum(out["capacity_rate_hot_kw_per_k"], out["capacity_rate_cold_noisy_kw_per_k"])

    out["energy_proxy_clean_kw"] = out["capacity_rate_cold_clean_kw_per_k"] * out["delta_t_in_clean_k"]
    out["energy_proxy_noisy_kw"] = out["capacity_rate_cold_noisy_kw_per_k"] * out["delta_t_in_noisy_k"]
    out["normalized_flow_ratio_clean"] = out["cold_inlet_mass_flow_kg_s"] / hot_mass_flow_kg_s
    out["normalized_flow_ratio_noisy"] = out["cold_inlet_mass_flow_kg_s_noisy"] / hot_mass_flow_kg_s

    out["thermal_diffusivity_hot_m2_s"] = k_hot / (rho_hot * cp_hot_j_kgk)
    out["thermal_diffusivity_cold_m2_s"] = k_cold / (rho_cold * cp_cold_j_kgk)
    out["prandtl_hot"] = mu_hot * cp_hot_j_kgk / k_hot
    out["prandtl_cold"] = mu_cold * cp_cold_j_kgk / k_cold

    out["delta_t_in_clean_sq"] = out["delta_t_in_clean_k"] ** 2
    out["delta_t_in_noisy_sq"] = out["delta_t_in_noisy_k"] ** 2
    out["flow_x_delta_clean"] = out["cold_inlet_mass_flow_kg_s"] * out["delta_t_in_clean_k"]
    out["flow_x_delta_noisy"] = out["cold_inlet_mass_flow_kg_s_noisy"] * out["delta_t_in_noisy_k"]
    out["temp_flow_interaction_noisy"] = out["hot_inlet_temperature_k_noisy"] * out["cold_inlet_mass_flow_kg_s_noisy"]
    return out

feature_df = build_feature_frame(df, CONFIG)
print("Feature frame shape:", feature_df.shape)
display(feature_df.head())


## Step 6: Define Training-Safe Features And Diagnostic Physics Columns

Training-safe features use only inlet-side information and assumed constants.


In [ ]:
training_safe_features = feature_df.columns.tolist()
research_targets = ["hx_1_heat_load_kw", "hot_outlet_temperature_k"]
print("Number of training-safe features:", len(training_safe_features))
print("Targets:", research_targets)
display(pd.DataFrame({"training_safe_feature": training_safe_features}))


## Step 7: Diagnostic Physics Features (Not Used For Training)

These features use outlet-side information only for interpretation and sanity checking.


In [ ]:
def build_diagnostic_physics(df_in, config):
    cold_inlet_temperature_k = config["cold_inlet_temperature_k"]
    hot_mass_flow = config["assumed_hot_mass_flow_kg_s"]
    cp_hot = config["cp_hot_kj_kgk"]
    cp_cold = config["cp_cold_kj_kgk"]

    diag = pd.DataFrame(index=df_in.index)
    diag["delta_t_in_clean_k"] = df_in["hot_inlet_temperature_k"] - cold_inlet_temperature_k
    diag["observed_q_hot_kw_signed"] = hot_mass_flow * cp_hot * (df_in["hot_inlet_temperature_k"] - df_in["hot_outlet_temperature_k"])
    diag["observed_q_hot_kw_abs"] = np.abs(diag["observed_q_hot_kw_signed"])
    diag["reported_heat_load_kw"] = df_in["hx_1_heat_load_kw"]

    c_hot = hot_mass_flow * cp_hot
    c_cold = df_in["cold_inlet_mass_flow_kg_s"] * cp_cold
    c_min = np.minimum(c_hot, c_cold)
    diag["effectiveness_proxy"] = df_in["hx_1_heat_load_kw"] / (c_min * diag["delta_t_in_clean_k"] + 1e-12)
    diag["heat_load_gap_kw"] = df_in["hx_1_heat_load_kw"] - diag["observed_q_hot_kw_signed"]
    return diag

diagnostic_df = build_diagnostic_physics(df, CONFIG)
display(diagnostic_df.head())
display(diagnostic_df.describe().T)


## Step 8: Exploratory Data Analysis

We visualize key input distributions, targets, scatter relationships, and correlations.


In [ ]:
selected_eda_columns = [
    "hot_inlet_temperature_k",
    "hot_inlet_temperature_k_noisy",
    "cold_inlet_mass_flow_kg_s",
    "cold_inlet_mass_flow_kg_s_noisy",
    "delta_t_in_clean_k",
    "delta_t_in_noisy_k",
    "capacity_rate_ratio_clean",
    "capacity_rate_ratio_noisy",
    "energy_proxy_clean_kw",
    "energy_proxy_noisy_kw",
]

eda_df = feature_df[selected_eda_columns].copy()
eda_df["hx_1_heat_load_kw"] = df["hx_1_heat_load_kw"]
eda_df["hot_outlet_temperature_k"] = df["hot_outlet_temperature_k"]

fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes = axes.ravel()
for idx, col in enumerate(eda_df.columns[:12]):
    axes[idx].hist(eda_df[col], bins=30, color="#2a6f97", alpha=0.85, edgecolor="black")
    axes[idx].set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
sample_for_scatter = eda_df.sample(min(len(eda_df), 1500), random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes[0, 0].scatter(sample_for_scatter["delta_t_in_clean_k"], sample_for_scatter["hx_1_heat_load_kw"], alpha=0.45)
axes[0, 0].set_title("Heat Load vs Clean Inlet Temperature Difference")
axes[0, 0].set_xlabel("delta_t_in_clean_k")
axes[0, 0].set_ylabel("hx_1_heat_load_kw")

axes[0, 1].scatter(sample_for_scatter["cold_inlet_mass_flow_kg_s"], sample_for_scatter["hx_1_heat_load_kw"], alpha=0.45)
axes[0, 1].set_title("Heat Load vs Cold Inlet Mass Flow")
axes[0, 1].set_xlabel("cold_inlet_mass_flow_kg_s")
axes[0, 1].set_ylabel("hx_1_heat_load_kw")

axes[1, 0].scatter(sample_for_scatter["delta_t_in_clean_k"], sample_for_scatter["hot_outlet_temperature_k"], alpha=0.45)
axes[1, 0].set_title("Hot Outlet Temperature vs Clean Inlet Temperature Difference")
axes[1, 0].set_xlabel("delta_t_in_clean_k")
axes[1, 0].set_ylabel("hot_outlet_temperature_k")

axes[1, 1].scatter(sample_for_scatter["energy_proxy_clean_kw"], sample_for_scatter["hx_1_heat_load_kw"], alpha=0.45)
axes[1, 1].set_title("Heat Load vs Energy Proxy")
axes[1, 1].set_xlabel("energy_proxy_clean_kw")
axes[1, 1].set_ylabel("hx_1_heat_load_kw")

plt.tight_layout()
plt.show()


In [ ]:
corr_columns = [
    "hot_inlet_temperature_k",
    "hot_inlet_temperature_k_noisy",
    "cold_inlet_mass_flow_kg_s",
    "cold_inlet_mass_flow_kg_s_noisy",
    "delta_t_in_clean_k",
    "delta_t_in_noisy_k",
    "capacity_rate_ratio_clean",
    "capacity_rate_ratio_noisy",
    "energy_proxy_clean_kw",
    "energy_proxy_noisy_kw",
    "hx_1_heat_load_kw",
    "hot_outlet_temperature_k",
]

corr_df = pd.concat([feature_df, df[["hx_1_heat_load_kw", "hot_outlet_temperature_k"]]], axis=1)[corr_columns]
plt.figure(figsize=(12, 9))
sns.heatmap(corr_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap For Key Engineered Features And Targets")
plt.show()


## Step 9: From-Scratch Utility Functions

We implement train/validation/test split, clipping-based outlier handling, standardization, and regression metrics.


In [ ]:
def train_val_test_split(X, y, test_size=0.2, val_fraction_of_train=0.2, random_state=42):
    rng = np.random.default_rng(random_state)
    indices = np.arange(len(X))
    rng.shuffle(indices)
    test_count = int(len(X) * test_size)
    test_idx = indices[:test_count]
    train_idx = indices[test_count:]
    val_count = int(len(train_idx) * val_fraction_of_train)
    val_idx = train_idx[:val_count]
    train_only_idx = train_idx[val_count:]
    return (
        X.iloc[train_only_idx].reset_index(drop=True),
        X.iloc[val_idx].reset_index(drop=True),
        X.iloc[test_idx].reset_index(drop=True),
        y.iloc[train_only_idx].reset_index(drop=True),
        y.iloc[val_idx].reset_index(drop=True),
        y.iloc[test_idx].reset_index(drop=True),
    )

class QuantileClipper:
    def __init__(self, lower_q=0.01, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q
        self.lower_bounds_ = None
        self.upper_bounds_ = None
    def fit(self, X):
        X_arr = np.asarray(X, dtype=float)
        self.lower_bounds_ = np.quantile(X_arr, self.lower_q, axis=0)
        self.upper_bounds_ = np.quantile(X_arr, self.upper_q, axis=0)
        return self
    def transform(self, X):
        X_arr = np.asarray(X, dtype=float)
        return np.clip(X_arr, self.lower_bounds_, self.upper_bounds_)
    def fit_transform(self, X):
        return self.fit(X).transform(X)

class StandardScalerScratch:
    def __init__(self):
        self.mean_ = None
        self.std_ = None
    def fit(self, X):
        X_arr = np.asarray(X, dtype=float)
        self.mean_ = X_arr.mean(axis=0)
        self.std_ = X_arr.std(axis=0)
        self.std_[self.std_ == 0] = 1.0
        return self
    def transform(self, X):
        X_arr = np.asarray(X, dtype=float)
        return (X_arr - self.mean_) / self.std_
    def fit_transform(self, X):
        return self.fit(X).transform(X)

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def r2_score_scratch(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - (ss_res / (ss_tot + 1e-12))

def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-12))) * 100.0

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "R2": r2_score_scratch(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
    }


## Step 10: From-Scratch Model Implementations

We implement a baseline, linear regression, linear epsilon-SVR, KNN, decision tree, random forest, and gradient boosting directly in numpy/custom code.


In [ ]:
class MeanRegressor:
    def fit(self, X, y):
        self.mean_ = float(np.mean(y))
        self.history_ = None
        return self
    def predict(self, X):
        return np.full(shape=(len(X),), fill_value=self.mean_, dtype=float)

class LinearRegressionGD:
    def __init__(self, lr=0.03, epochs=1500, l2=0.0):
        self.lr = lr
        self.epochs = epochs
        self.l2 = l2
        self.w = None
        self.b = 0.0
        self.history_ = []
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features, dtype=float)
        self.b = 0.0
        self.history_ = []
        for epoch in range(self.epochs):
            y_pred = X @ self.w + self.b
            error = y_pred - y
            grad_w = (2.0 / n_samples) * (X.T @ error) + 2.0 * self.l2 * self.w
            grad_b = (2.0 / n_samples) * np.sum(error)
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b
            if epoch % 25 == 0 or epoch == self.epochs - 1:
                loss = np.mean(error ** 2) + self.l2 * np.sum(self.w ** 2)
                self.history_.append((epoch, float(loss)))
        return self
    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b

class LinearSVRSubgradient:
    def __init__(self, lr=0.01, epochs=1200, epsilon=0.5, C=1.0, l2=0.0):
        self.lr = lr
        self.epochs = epochs
        self.epsilon = epsilon
        self.C = C
        self.l2 = l2
        self.w = None
        self.b = 0.0
        self.history_ = []
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features, dtype=float)
        self.b = 0.0
        self.history_ = []
        for epoch in range(self.epochs):
            y_pred = X @ self.w + self.b
            residual = y_pred - y
            abs_res = np.abs(residual)
            active = abs_res > self.epsilon
            sign = np.sign(residual)
            if np.any(active):
                grad_w_loss = self.C * np.mean((sign[active, None] * X[active]), axis=0)
                grad_b_loss = self.C * np.mean(sign[active])
            else:
                grad_w_loss = np.zeros(n_features, dtype=float)
                grad_b_loss = 0.0
            grad_w = grad_w_loss + 2.0 * self.l2 * self.w
            grad_b = grad_b_loss
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b
            if epoch % 25 == 0 or epoch == self.epochs - 1:
                epsilon_loss = np.maximum(0.0, abs_res - self.epsilon).mean()
                objective = epsilon_loss + self.l2 * np.sum(self.w ** 2)
                self.history_.append((epoch, float(objective)))
        return self
    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b

class KNNRegressorScratch:
    def __init__(self, k=7, weighted=True):
        self.k = k
        self.weighted = weighted
        self.X_train = None
        self.y_train = None
        self.history_ = None
    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y, dtype=float)
        return self
    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = []
        for row in X:
            distances = np.sqrt(np.sum((self.X_train - row) ** 2, axis=1))
            neighbor_idx = np.argsort(distances)[: self.k]
            neighbor_targets = self.y_train[neighbor_idx]
            if self.weighted:
                weights = 1.0 / (distances[neighbor_idx] + 1e-8)
                pred = np.sum(weights * neighbor_targets) / np.sum(weights)
            else:
                pred = np.mean(neighbor_targets)
            preds.append(pred)
        return np.array(preds, dtype=float)


In [ ]:
class TreeNode:
    def __init__(self, prediction=None, feature_index=None, threshold=None, left=None, right=None):
        self.prediction = prediction
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right

class DecisionTreeRegressorScratch:
    def __init__(self, max_depth=6, min_samples_split=20, min_samples_leaf=8, max_features=None, n_thresholds=18, random_state=42):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_thresholds = n_thresholds
        self.random_state = random_state
        self.root = None
        self.feature_importance_splits_ = None
        self._rng = np.random.default_rng(random_state)
        self.history_ = None
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.n_features_ = X.shape[1]
        self.feature_importance_splits_ = np.zeros(self.n_features_, dtype=float)
        self.root = self._grow_tree(X, y, depth=0)
        return self
    def _variance(self, y):
        return 0.0 if len(y) == 0 else float(np.var(y))
    def _candidate_features(self):
        all_features = np.arange(self.n_features_)
        if self.max_features is None or self.max_features >= self.n_features_:
            return all_features
        return self._rng.choice(all_features, size=self.max_features, replace=False)
    def _best_split(self, X, y):
        best_feature, best_threshold, best_score = None, None, np.inf
        if self._variance(y) <= 1e-12:
            return None, None
        for feature_idx in self._candidate_features():
            column = X[:, feature_idx]
            unique_vals = np.unique(column)
            if len(unique_vals) <= 1:
                continue
            if len(unique_vals) > self.n_thresholds:
                thresholds = np.unique(np.quantile(column, np.linspace(0.05, 0.95, self.n_thresholds)))
            else:
                thresholds = unique_vals[:-1]
            for threshold in thresholds:
                left_mask = column <= threshold
                right_mask = ~left_mask
                if left_mask.sum() < self.min_samples_leaf or right_mask.sum() < self.min_samples_leaf:
                    continue
                y_left, y_right = y[left_mask], y[right_mask]
                weighted_var = (len(y_left) * self._variance(y_left) + len(y_right) * self._variance(y_right)) / len(y)
                if weighted_var < best_score:
                    best_score, best_feature, best_threshold = weighted_var, feature_idx, threshold
        return best_feature, best_threshold
    def _grow_tree(self, X, y, depth):
        node = TreeNode(prediction=float(np.mean(y)))
        if depth >= self.max_depth or len(y) < self.min_samples_split or self._variance(y) <= 1e-12:
            return node
        feature_idx, threshold = self._best_split(X, y)
        if feature_idx is None:
            return node
        left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask
        if left_mask.sum() < self.min_samples_leaf or right_mask.sum() < self.min_samples_leaf:
            return node
        self.feature_importance_splits_[feature_idx] += 1.0
        node.feature_index = feature_idx
        node.threshold = threshold
        node.left = self._grow_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._grow_tree(X[right_mask], y[right_mask], depth + 1)
        return node
    def _predict_row(self, row, node):
        if node.feature_index is None:
            return node.prediction
        return self._predict_row(row, node.left if row[node.feature_index] <= node.threshold else node.right)
    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_row(row, self.root) for row in X], dtype=float)

class RandomForestRegressorScratch:
    def __init__(self, n_estimators=15, max_depth=6, min_samples_split=20, min_samples_leaf=8, max_features="sqrt", n_thresholds=18, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_thresholds = n_thresholds
        self.random_state = random_state
        self.trees = []
        self.feature_importances_ = None
        self.history_ = None
    def _resolve_max_features(self, n_features):
        if self.max_features == "sqrt":
            return max(1, int(np.sqrt(n_features)))
        if self.max_features == "log2":
            return max(1, int(np.log2(n_features)))
        if isinstance(self.max_features, int):
            return min(n_features, self.max_features)
        return n_features
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape
        rng = np.random.default_rng(self.random_state)
        self.trees = []
        importances = np.zeros(n_features, dtype=float)
        max_features = self._resolve_max_features(n_features)
        for est_idx in range(self.n_estimators):
            bootstrap_idx = rng.choice(np.arange(n_samples), size=n_samples, replace=True)
            tree = DecisionTreeRegressorScratch(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=max_features,
                n_thresholds=self.n_thresholds,
                random_state=self.random_state + est_idx,
            )
            tree.fit(X[bootstrap_idx], y[bootstrap_idx])
            self.trees.append(tree)
            importances += tree.feature_importance_splits_
        self.feature_importances_ = importances / importances.sum() if importances.sum() != 0 else importances
        return self
    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.mean(np.array([tree.predict(X) for tree in self.trees]), axis=0)

class GradientBoostingRegressorScratch:
    def __init__(self, n_estimators=35, learning_rate=0.08, max_depth=2, min_samples_split=20, min_samples_leaf=8, n_thresholds=16, random_state=42):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.n_thresholds = n_thresholds
        self.random_state = random_state
        self.base_value_ = None
        self.trees_ = []
        self.history_ = []
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.base_value_ = float(np.mean(y))
        current_pred = np.full_like(y, self.base_value_, dtype=float)
        self.trees_, self.history_ = [], []
        for i in range(self.n_estimators):
            residual = y - current_pred
            tree = DecisionTreeRegressorScratch(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=None,
                n_thresholds=self.n_thresholds,
                random_state=self.random_state + i,
            )
            tree.fit(X, residual)
            update = tree.predict(X)
            current_pred = current_pred + self.learning_rate * update
            self.trees_.append(tree)
            if i % 2 == 0 or i == self.n_estimators - 1:
                self.history_.append((i, float(np.mean((y - current_pred) ** 2))))
        return self
    def predict(self, X):
        X = np.asarray(X, dtype=float)
        pred = np.full(shape=(len(X),), fill_value=self.base_value_, dtype=float)
        for tree in self.trees_:
            pred += self.learning_rate * tree.predict(X)
        return pred


## Step 11: Experiment Driver

This section sets up the full experiment loop with validation-based model selection and final test evaluation.


In [ ]:
MODEL_SEARCH_SPACE = {
    "MeanBaseline": [{}],
    "LinearRegressionGD": [
        {"lr": 0.03, "epochs": 1200, "l2": 0.0},
        {"lr": 0.02, "epochs": 1600, "l2": 1e-4},
    ],
    "LinearSVR": [
        {"lr": 0.01, "epochs": 1000, "epsilon": 0.35, "C": 0.8, "l2": 1e-4},
        {"lr": 0.008, "epochs": 1300, "epsilon": 0.50, "C": 1.0, "l2": 1e-4},
    ],
    "KNN": [
        {"k": 5, "weighted": True},
        {"k": 9, "weighted": True},
        {"k": 11, "weighted": False},
    ],
    "DecisionTree": [
        {"max_depth": 4, "min_samples_split": 25, "min_samples_leaf": 10, "max_features": None, "n_thresholds": 14, "random_state": 42},
        {"max_depth": 6, "min_samples_split": 25, "min_samples_leaf": 10, "max_features": None, "n_thresholds": 18, "random_state": 42},
    ],
    "RandomForest": [
        {"n_estimators": 10, "max_depth": 5, "min_samples_split": 25, "min_samples_leaf": 10, "max_features": "sqrt", "n_thresholds": 14, "random_state": 42},
        {"n_estimators": 16, "max_depth": 6, "min_samples_split": 25, "min_samples_leaf": 10, "max_features": "sqrt", "n_thresholds": 18, "random_state": 42},
    ],
    "GradientBoosting": [
        {"n_estimators": 25, "learning_rate": 0.08, "max_depth": 2, "min_samples_split": 25, "min_samples_leaf": 10, "n_thresholds": 14, "random_state": 42},
        {"n_estimators": 40, "learning_rate": 0.06, "max_depth": 2, "min_samples_split": 25, "min_samples_leaf": 10, "n_thresholds": 16, "random_state": 42},
    ],
}

def build_model(model_name, params):
    if model_name == "MeanBaseline":
        return MeanRegressor()
    if model_name == "LinearRegressionGD":
        return LinearRegressionGD(**params)
    if model_name == "LinearSVR":
        return LinearSVRSubgradient(**params)
    if model_name == "KNN":
        return KNNRegressorScratch(**params)
    if model_name == "DecisionTree":
        return DecisionTreeRegressorScratch(**params)
    if model_name == "RandomForest":
        return RandomForestRegressorScratch(**params)
    if model_name == "GradientBoosting":
        return GradientBoostingRegressorScratch(**params)
    raise ValueError(f"Unknown model name: {model_name}")

def prepare_data_for_target(feature_frame, full_df, target_column, config):
    X = feature_frame.copy()
    y = full_df[target_column].copy()
    X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(
        X, y, test_size=config["test_size"], val_fraction_of_train=config["validation_fraction_of_train"], random_state=config["random_state"]
    )
    clipper = QuantileClipper(lower_q=0.01, upper_q=0.99)
    scaler = StandardScalerScratch()
    X_train_proc = scaler.fit_transform(clipper.fit_transform(X_train.values))
    X_val_proc = scaler.transform(clipper.transform(X_val.values))
    X_test_proc = scaler.transform(clipper.transform(X_test.values))
    return {
        "X_train": X_train, "X_val": X_val, "X_test": X_test,
        "y_train": y_train, "y_val": y_val, "y_test": y_test,
        "X_train_proc": X_train_proc, "X_val_proc": X_val_proc, "X_test_proc": X_test_proc,
        "clipper": clipper, "scaler": scaler, "feature_columns": X.columns.tolist(),
    }

def refit_on_train_and_val(model_name, best_params, prep_bundle):
    X_combined = np.vstack([prep_bundle["X_train_proc"], prep_bundle["X_val_proc"]])
    y_combined = np.concatenate([prep_bundle["y_train"].values, prep_bundle["y_val"].values])
    model = build_model(model_name, best_params)
    model.fit(X_combined, y_combined)
    return model

def run_experiment(feature_frame, full_df, target_column, config):
    prep = prepare_data_for_target(feature_frame, full_df, target_column, config)
    all_metrics, tuning_rows = [], []
    predictions = pd.DataFrame({"Actual": prep["y_test"].values})
    model_objects = {}
    for model_name, candidate_list in MODEL_SEARCH_SPACE.items():
        best_rmse, best_params = np.inf, None
        for params in candidate_list:
            model = build_model(model_name, params)
            model.fit(prep["X_train_proc"], prep["y_train"].values)
            val_pred = model.predict(prep["X_val_proc"])
            val_rmse = rmse(prep["y_val"].values, val_pred)
            tuning_rows.append({"Target": target_column, "Model": model_name, "Params": str(params), "Validation_RMSE": val_rmse})
            if val_rmse < best_rmse:
                best_rmse, best_params = val_rmse, params
        final_model = refit_on_train_and_val(model_name, best_params, prep)
        test_pred = final_model.predict(prep["X_test_proc"])
        metrics = regression_metrics(prep["y_test"].values, test_pred)
        all_metrics.append({"Target": target_column, "Model": model_name, "Best Params": str(best_params), **metrics})
        predictions[f"{model_name}_Predicted"] = test_pred
        model_objects[model_name] = final_model
    metrics_df = pd.DataFrame(all_metrics).sort_values("RMSE").reset_index(drop=True)
    tuning_df = pd.DataFrame(tuning_rows).sort_values(["Model", "Validation_RMSE"]).reset_index(drop=True)
    best_model_name = metrics_df.iloc[0]["Model"]
    artifact = {
        "target": target_column,
        "best_model_name": best_model_name,
        "best_model_object": model_objects[best_model_name],
        "all_models": model_objects,
        "clipper": prep["clipper"],
        "scaler": prep["scaler"],
        "feature_columns": prep["feature_columns"],
        "config": config,
    }
    return {
        "metrics_df": metrics_df,
        "predictions_df": predictions,
        "tuning_df": tuning_df,
        "artifact": artifact,
        "y_test": prep["y_test"].values,
        "X_test_original": prep["X_test"],
    }


## Step 12: Run The Primary Target Experiment

Primary target: `hx_1_heat_load_kw`


In [ ]:
primary_target = "hx_1_heat_load_kw"
primary_results = run_experiment(feature_df[training_safe_features], df, primary_target, CONFIG)
print("Primary target metrics:")
display(primary_results["metrics_df"].round(6))
print("Primary target tuning summary:")
display(primary_results["tuning_df"].head(20))


## Step 13: Run The Secondary Target Experiment

Secondary target: `hot_outlet_temperature_k`


In [ ]:
secondary_target = "hot_outlet_temperature_k"
secondary_results = run_experiment(feature_df[training_safe_features], df, secondary_target, CONFIG)
print("Secondary target metrics:")
display(secondary_results["metrics_df"].round(6))
print("Secondary target tuning summary:")
display(secondary_results["tuning_df"].head(20))


## Step 14: Metrics Comparison Plots


In [ ]:
def plot_metric_bars(metrics_df, title_prefix):
    metric_names = ["MAE", "RMSE", "R2", "MAPE"]
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.ravel()
    for idx, metric_name in enumerate(metric_names):
        ordered = metrics_df.sort_values(metric_name, ascending=(metric_name != "R2"))
        axes[idx].bar(ordered["Model"], ordered[metric_name], color="#468faf")
        axes[idx].set_title(f"{title_prefix} - {metric_name}")
        axes[idx].tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

plot_metric_bars(primary_results["metrics_df"], "Primary Target: Heat Load")
plot_metric_bars(secondary_results["metrics_df"], "Secondary Target: Hot Outlet Temperature")


## Step 15: Actual vs Predicted Plots


In [ ]:
def plot_actual_vs_predicted(results, title):
    pred_df = results["predictions_df"]
    actual = pred_df["Actual"].values
    model_cols = [c for c in pred_df.columns if c.endswith("_Predicted")]
    n_models = len(model_cols)
    n_cols = 3
    n_rows = int(np.ceil(n_models / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, model_cols):
        pred = pred_df[col].values
        ax.scatter(actual, pred, alpha=0.45)
        min_v = min(actual.min(), pred.min())
        max_v = max(actual.max(), pred.max())
        ax.plot([min_v, max_v], [min_v, max_v], "r--")
        ax.set_title(col.replace("_Predicted", ""))
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
    for ax in axes[n_models:]:
        ax.axis("off")
    plt.suptitle(title, y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()

plot_actual_vs_predicted(primary_results, "Parity Plots: Primary Target")
plot_actual_vs_predicted(secondary_results, "Parity Plots: Secondary Target")


## Step 16: Residual Diagnostics


In [ ]:
def plot_residual_diagnostics(results, title):
    pred_df = results["predictions_df"]
    actual = pred_df["Actual"].values
    model_cols = [c for c in pred_df.columns if c.endswith("_Predicted")]
    n_models = len(model_cols)
    fig, axes = plt.subplots(n_models, 2, figsize=(14, 4 * n_models))
    if n_models == 1:
        axes = np.array([axes])
    for row_idx, col in enumerate(model_cols):
        pred = pred_df[col].values
        residual = actual - pred
        axes[row_idx, 0].scatter(pred, residual, alpha=0.45)
        axes[row_idx, 0].axhline(0, color="red", linestyle="--")
        axes[row_idx, 0].set_title(f"{col} - Residual vs Predicted")
        axes[row_idx, 0].set_xlabel("Predicted")
        axes[row_idx, 0].set_ylabel("Residual")
        axes[row_idx, 1].hist(residual, bins=30, color="#2a9d8f", edgecolor="black")
        axes[row_idx, 1].set_title(f"{col} - Residual Histogram")
        axes[row_idx, 1].set_xlabel("Residual")
    plt.suptitle(title, y=1.01, fontsize=16)
    plt.tight_layout()
    plt.show()

plot_residual_diagnostics(primary_results, "Residual Diagnostics: Primary Target")
plot_residual_diagnostics(secondary_results, "Residual Diagnostics: Secondary Target")


## Step 17: Training Curves For Iterative Models


In [ ]:
def fit_iterative_examples_for_history(feature_frame, full_df, target_column, config):
    prep = prepare_data_for_target(feature_frame, full_df, target_column, config)
    lin = LinearRegressionGD(lr=0.02, epochs=1200, l2=1e-4)
    lin.fit(prep["X_train_proc"], prep["y_train"].values)
    svr = LinearSVRSubgradient(lr=0.008, epochs=1200, epsilon=0.5, C=1.0, l2=1e-4)
    svr.fit(prep["X_train_proc"], prep["y_train"].values)
    gb = GradientBoostingRegressorScratch(n_estimators=35, learning_rate=0.06, max_depth=2, min_samples_split=25, min_samples_leaf=10, n_thresholds=16, random_state=42)
    gb.fit(prep["X_train_proc"], prep["y_train"].values)
    return lin.history_, svr.history_, gb.history_

lin_hist, svr_hist, gb_hist = fit_iterative_examples_for_history(feature_df[training_safe_features], df, primary_target, CONFIG)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
epochs, losses = zip(*lin_hist)
plt.plot(epochs, losses, marker="o")
plt.title("Linear Regression Training Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.subplot(1, 3, 2)
epochs, losses = zip(*svr_hist)
plt.plot(epochs, losses, marker="o")
plt.title("Linear SVR Training Curve")
plt.xlabel("Epoch")
plt.ylabel("Objective")

plt.subplot(1, 3, 3)
epochs, losses = zip(*gb_hist)
plt.plot(epochs, losses, marker="o")
plt.title("Gradient Boosting Training Curve")
plt.xlabel("Boosting Iteration")
plt.ylabel("Training MSE")
plt.tight_layout()
plt.show()


## Step 18: Model Interpretation With Permutation Importance


In [ ]:
def permutation_importance(model, X_df, y_true, clipper, scaler, metric_func=rmse, random_state=42):
    rng = np.random.default_rng(random_state)
    base_X = scaler.transform(clipper.transform(X_df.values))
    base_pred = model.predict(base_X)
    base_score = metric_func(y_true, base_pred)
    rows = []
    for col in X_df.columns:
        X_perm = X_df.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        X_perm_proc = scaler.transform(clipper.transform(X_perm.values))
        perm_pred = model.predict(X_perm_proc)
        perm_score = metric_func(y_true, perm_pred)
        rows.append({"feature": col, "importance_rmse_increase": perm_score - base_score})
    return pd.DataFrame(rows).sort_values("importance_rmse_increase", ascending=False).reset_index(drop=True)

primary_best_model_name = primary_results["artifact"]["best_model_name"]
primary_best_model = primary_results["artifact"]["best_model_object"]
primary_importance_df = permutation_importance(primary_best_model, primary_results["X_test_original"], primary_results["y_test"], primary_results["artifact"]["clipper"], primary_results["artifact"]["scaler"])

secondary_best_model_name = secondary_results["artifact"]["best_model_name"]
secondary_best_model = secondary_results["artifact"]["best_model_object"]
secondary_importance_df = permutation_importance(secondary_best_model, secondary_results["X_test_original"], secondary_results["y_test"], secondary_results["artifact"]["clipper"], secondary_results["artifact"]["scaler"])

print("Primary best model:", primary_best_model_name)
display(primary_importance_df.head(12))
print("Secondary best model:", secondary_best_model_name)
display(secondary_importance_df.head(12))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_primary = primary_importance_df.head(10).iloc[::-1]
axes[0].barh(top_primary["feature"], top_primary["importance_rmse_increase"], color="#f4a261")
axes[0].set_title(f"Permutation Importance - {primary_best_model_name} - Heat Load")
axes[0].set_xlabel("RMSE increase after permutation")

top_secondary = secondary_importance_df.head(10).iloc[::-1]
axes[1].barh(top_secondary["feature"], top_secondary["importance_rmse_increase"], color="#e76f51")
axes[1].set_title(f"Permutation Importance - {secondary_best_model_name} - Hot Outlet Temperature")
axes[1].set_xlabel("RMSE increase after permutation")

plt.tight_layout()
plt.show()


## Step 19: Model Ranking Summary Tables


In [ ]:
primary_ranked = primary_results["metrics_df"].copy()
secondary_ranked = secondary_results["metrics_df"].copy()
primary_ranked["Rank_by_RMSE"] = np.arange(1, len(primary_ranked) + 1)
secondary_ranked["Rank_by_RMSE"] = np.arange(1, len(secondary_ranked) + 1)
display(primary_ranked.round(6))
display(secondary_ranked.round(6))


## Step 20: Compare Measured Heat Load Against A Physics Reconstruction

This is a diagnostic exercise only. We compare the reported heat load with a hot-side reconstruction using m * Cp * DeltaT.


In [ ]:
diag_plot_df = diagnostic_df.sample(min(len(diagnostic_df), 1500), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].scatter(diag_plot_df["reported_heat_load_kw"], diag_plot_df["observed_q_hot_kw_signed"], alpha=0.45)
axes[0].set_title("Reported Heat Load vs Hot-Side Reconstructed Heat Load")
axes[0].set_xlabel("reported_heat_load_kw")
axes[0].set_ylabel("observed_q_hot_kw_signed")
axes[1].hist(diagnostic_df["effectiveness_proxy"], bins=30, color="#588157", edgecolor="black")
axes[1].set_title("Distribution of Effectiveness Proxy")
axes[1].set_xlabel("effectiveness_proxy")
plt.tight_layout()
plt.show()

display(diagnostic_df[["reported_heat_load_kw", "observed_q_hot_kw_signed", "observed_q_hot_kw_abs", "effectiveness_proxy", "heat_load_gap_kw"]].describe().T)


## Step 21: Physics-Informed LSTM Integration

Now we extend the notebook with a Physics-Informed Long Short-Term Memory (PI-LSTM) model.

### Why add PI-LSTM after traditional ML?

The traditional ML section already gives us a strong tabular baseline. That remains important.

We still add PI-LSTM because heat-exchanger behavior is naturally sequential, and PI-LSTM gives us a way to model:

1. temporal dependencies across operating steps
2. small-data learning with ordered sequences
3. physics-aware output behavior through constrained loss design

For Version 1, PI-LSTM should be presented as an advanced comparison model, not automatically as the overall winner on this dataset.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    TENSORFLOW_AVAILABLE = True
    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("TensorFlow not available. Install with: pip install tensorflow")


### Physics-Informed LSTM Model Class

This implementation includes:
- custom physics-informed loss
- temperature-bound penalty terms
- sequence-based learning for temporal patterns
- stacked LSTM layers with dropout regularization

In [ ]:
if TENSORFLOW_AVAILABLE:
    class PhysicsInformedLSTM:
        def __init__(self, sequence_length=10, lstm_units=64, learning_rate=0.001):
            self.sequence_length = sequence_length
            self.lstm_units = lstm_units
            self.learning_rate = learning_rate
            self.model = None
            self.scaler_X = StandardScaler()
            self.scaler_y = StandardScaler()
            self.history = None
        
        def create_sequences(self, X, y):
            X_seq, y_seq = [], []
            for i in range(len(X) - self.sequence_length):
                X_seq.append(X[i:i + self.sequence_length])
                y_seq.append(y[i + self.sequence_length])
            return np.array(X_seq), np.array(y_seq)
        
        def physics_loss(self, y_true, y_pred):
            mse_loss = tf.reduce_mean(tf.square(y_true - y_pred))
            hot_outlet_pred = y_pred[:, 0]
            cold_outlet_pred = y_pred[:, 1]
            physics_penalty = tf.reduce_mean(
                tf.maximum(0.0, -hot_outlet_pred) +
                tf.maximum(0.0, -cold_outlet_pred)
            )
            total_loss = mse_loss + 0.1 * physics_penalty
            return total_loss
        
        def build_model(self, input_shape):
            inputs = layers.Input(shape=input_shape)
            x = layers.LSTM(self.lstm_units, return_sequences=True)(inputs)
            x = layers.Dropout(0.2)(x)
            x = layers.LSTM(self.lstm_units // 2, return_sequences=True)(x)
            x = layers.Dropout(0.2)(x)
            x = layers.LSTM(self.lstm_units // 4, return_sequences=False)(x)
            x = layers.Dropout(0.2)(x)
            x = layers.Dense(32, activation='relu')(x)
            x = layers.Dense(16, activation='relu')(x)
            outputs = layers.Dense(2, activation='linear')(x)
            model = keras.Model(inputs=inputs, outputs=outputs)
            optimizer = keras.optimizers.Adam(learning_rate=self.learning_rate)
            model.compile(optimizer=optimizer, loss=self.physics_loss, metrics=['mae', 'mse'])
            self.model = model
            return model
        
        def prepare_data(self, df_in):
            input_features = [
                'hot_inlet_temperature_k',
                'cold_inlet_mass_flow_kg_s',
                'hx_1_heat_load_kw',
                'hot_outlet_pressure_pa',
                'cold_outlet_pressure_pa',
                'hot_outlet_mass_flow_kg_s',
                'cold_outlet_mass_flow_kg_s',
                'hx_1_logarithmic_mean_temperature_difference_lmtd_k'
            ]
            output_features = [
                'hot_outlet_temperature_k',
                'cold_outlet_temperature_k'
            ]
            X = df_in[input_features].values
            y = df_in[output_features].values
            return X, y
        
        def train(self, X_train, y_train, X_val, y_val, epochs=100, batch_size=32):
            X_train_scaled = self.scaler_X.fit_transform(X_train.reshape(-1, X_train.shape[-1]))
            X_train_scaled = X_train_scaled.reshape(X_train.shape)
            X_val_scaled = self.scaler_X.transform(X_val.reshape(-1, X_val.shape[-1]))
            X_val_scaled = X_val_scaled.reshape(X_val.shape)
            y_train_scaled = self.scaler_y.fit_transform(y_train)
            y_val_scaled = self.scaler_y.transform(y_val)
            early_stopping = keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=15,
                restore_best_weights=True
            )
            reduce_lr = keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-6
            )
            self.history = self.model.fit(
                X_train_scaled, y_train_scaled,
                validation_data=(X_val_scaled, y_val_scaled),
                epochs=epochs,
                batch_size=batch_size,
                callbacks=[early_stopping, reduce_lr],
                verbose=1
            )
            return self.history
        
        def predict(self, X):
            X_scaled = self.scaler_X.transform(X.reshape(-1, X.shape[-1]))
            X_scaled = X_scaled.reshape(X.shape)
            y_pred_scaled = self.model.predict(X_scaled, verbose=0)
            y_pred = self.scaler_y.inverse_transform(y_pred_scaled)
            return y_pred
        
        def evaluate(self, X_test, y_test):
            y_pred = self.predict(X_test)
            mae_hot = np.mean(np.abs(y_test[:, 0] - y_pred[:, 0]))
            mae_cold = np.mean(np.abs(y_test[:, 1] - y_pred[:, 1]))
            rmse_hot = np.sqrt(np.mean((y_test[:, 0] - y_pred[:, 0])**2))
            rmse_cold = np.sqrt(np.mean((y_test[:, 1] - y_pred[:, 1])**2))
            mape_hot = np.mean(np.abs((y_test[:, 0] - y_pred[:, 0]) / y_test[:, 0])) * 100
            mape_cold = np.mean(np.abs((y_test[:, 1] - y_pred[:, 1]) / y_test[:, 1])) * 100
            r2_hot = r2_score_scratch(y_test[:, 0], y_pred[:, 0])
            r2_cold = r2_score_scratch(y_test[:, 1], y_pred[:, 1])
            results = {
                'hot_outlet': {
                    'MAE': mae_hot,
                    'RMSE': rmse_hot,
                    'MAPE': mape_hot,
                    'R2': r2_hot
                },
                'cold_outlet': {
                    'MAE': mae_cold,
                    'RMSE': rmse_cold,
                    'MAPE': mape_cold,
                    'R2': r2_cold
                }
            }
            return results, y_pred
    
    print("Physics-Informed LSTM class defined successfully")
else:
    print("Skipping PI-LSTM implementation (TensorFlow not available)")


### Train Physics-Informed LSTM Model

We train the PI-LSTM on the available sequential process variables.

Important presentation note:
- the hot outlet temperature is the main trustworthy PI-LSTM target for this dataset
- the cold outlet column is not reliable enough to support strong conclusions

In [ ]:
if TENSORFLOW_AVAILABLE:
    print("Initializing Physics-Informed LSTM...")
    pi_lstm = PhysicsInformedLSTM(
        sequence_length=10,
        lstm_units=64,
        learning_rate=0.001
    )
    
    print("Preparing data for PI-LSTM...")
    X_lstm, y_lstm = pi_lstm.prepare_data(df)
    print(f"Input shape: {X_lstm.shape}")
    print(f"Output shape: {y_lstm.shape}")
    
    X_seq, y_seq = pi_lstm.create_sequences(X_lstm, y_lstm)
    print(f"Sequence input shape: {X_seq.shape}")
    print(f"Sequence output shape: {y_seq.shape}")
    
    X_temp, X_test_lstm, y_temp, y_test_lstm = train_test_split(
        X_seq, y_seq, test_size=0.15, random_state=42, shuffle=False
    )
    X_train_lstm, X_val_lstm, y_train_lstm, y_val_lstm = train_test_split(
        X_temp, y_temp, test_size=0.176, random_state=42, shuffle=False
    )
    
    print(f"Train set: {X_train_lstm.shape[0]} samples")
    print(f"Validation set: {X_val_lstm.shape[0]} samples")
    print(f"Test set: {X_test_lstm.shape[0]} samples")
    
    print("Building model architecture...")
    model_lstm = pi_lstm.build_model(input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2]))
    model_lstm.summary()
    
    print("Training Physics-Informed LSTM...")
    history_lstm = pi_lstm.train(
        X_train_lstm, y_train_lstm,
        X_val_lstm, y_val_lstm,
        epochs=100,
        batch_size=32
    )
    
    print("Evaluating on test set...")
    results_lstm, y_pred_lstm = pi_lstm.evaluate(X_test_lstm, y_test_lstm)
    
    print("\nTest Set Performance:")
    print("-" * 50)
    print("Hot Outlet Temperature:")
    print(f"  MAE:  {results_lstm['hot_outlet']['MAE']:.4f} K")
    print(f"  RMSE: {results_lstm['hot_outlet']['RMSE']:.4f} K")
    print(f"  MAPE: {results_lstm['hot_outlet']['MAPE']:.4f} %")
    print(f"  R2:   {results_lstm['hot_outlet']['R2']:.4f}")
    print("\nCold Outlet Temperature:")
    print(f"  MAE:  {results_lstm['cold_outlet']['MAE']:.4f} K")
    print(f"  RMSE: {results_lstm['cold_outlet']['RMSE']:.4f} K")
    print(f"  MAPE: {results_lstm['cold_outlet']['MAPE']:.4f} %")
    print(f"  R2:   {results_lstm['cold_outlet']['R2']:.4f}")
else:
    print("Skipping PI-LSTM training (TensorFlow not available)")


### Visualize PI-LSTM Training History

In [ ]:
if TENSORFLOW_AVAILABLE and 'history_lstm' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    axes[0].plot(history_lstm.history['loss'], label='Training Loss')
    axes[0].plot(history_lstm.history['val_loss'], label='Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('PI-LSTM Training Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(history_lstm.history['mae'], label='Training MAE')
    axes[1].plot(history_lstm.history['val_mae'], label='Validation MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].set_title('PI-LSTM Mean Absolute Error')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()


### PI-LSTM Prediction Visualization

The hot-outlet plot is the main presentation-grade visualization.
The cold-outlet plot should be treated as diagnostic only because the dataset contains problematic cold-outlet values.

In [ ]:
if TENSORFLOW_AVAILABLE and 'y_pred_lstm' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    axes[0].scatter(y_test_lstm[:, 0], y_pred_lstm[:, 0], alpha=0.5, s=10)
    axes[0].plot([y_test_lstm[:, 0].min(), y_test_lstm[:, 0].max()],
                 [y_test_lstm[:, 0].min(), y_test_lstm[:, 0].max()],
                 'r--', lw=2, label='Perfect Prediction')
    axes[0].set_xlabel('Actual Hot Outlet Temperature (K)')
    axes[0].set_ylabel('Predicted Hot Outlet Temperature (K)')
    axes[0].set_title('PI-LSTM: Hot Outlet Temperature')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].scatter(y_test_lstm[:, 1], y_pred_lstm[:, 1], alpha=0.5, s=10)
    axes[1].plot([y_test_lstm[:, 1].min(), y_test_lstm[:, 1].max()],
                 [y_test_lstm[:, 1].min(), y_test_lstm[:, 1].max()],
                 'r--', lw=2, label='Perfect Prediction')
    axes[1].set_xlabel('Actual Cold Outlet Temperature (K)')
    axes[1].set_ylabel('Predicted Cold Outlet Temperature (K)')
    axes[1].set_title('PI-LSTM: Cold Outlet Temperature (Diagnostic Only)')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()


### Model Comparison: Traditional ML vs PI-LSTM

This comparison should focus mainly on hot outlet temperature prediction.
That keeps the comparison honest and aligned with the valid part of the dataset.

In [ ]:
if TENSORFLOW_AVAILABLE and 'results_lstm' in locals():
    comparison_data = []
    
    for _, row in secondary_results['metrics_df'].iterrows():
        comparison_data.append({
            'Model': row['Model'],
            'Type': 'Traditional ML',
            'Target': 'Hot Outlet Temp',
            'MAE': row['MAE'],
            'RMSE': row['RMSE'],
            'R2': row['R2'],
            'MAPE': row['MAPE']
        })
    
    comparison_data.append({
        'Model': 'PI-LSTM',
        'Type': 'Deep Learning',
        'Target': 'Hot Outlet Temp',
        'MAE': results_lstm['hot_outlet']['MAE'],
        'RMSE': results_lstm['hot_outlet']['RMSE'],
        'R2': results_lstm['hot_outlet']['R2'],
        'MAPE': results_lstm['hot_outlet']['MAPE']
    })
    
    comparison_df = pd.DataFrame(comparison_data).sort_values('RMSE')
    
    print("Model Comparison for Hot Outlet Temperature Prediction:")
    display(comparison_df.round(4))
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    metrics = ['MAE', 'RMSE', 'R2', 'MAPE']
    for idx, metric in enumerate(metrics):
        ax = axes[idx // 2, idx % 2]
        sorted_df = comparison_df.sort_values(metric, ascending=(metric != 'R2'))
        colors = ['#e76f51' if t == 'Deep Learning' else '#468faf' for t in sorted_df['Type']]
        ax.barh(sorted_df['Model'], sorted_df[metric], color=colors)
        ax.set_title(f'{metric} Comparison')
        ax.set_xlabel(metric)
    
    plt.tight_layout()
    plt.show()


## Summary and Conclusions

### Traditional ML Models
- From-scratch traditional ML models establish the main baseline for this project
- `LinearRegressionGD` is the strongest saved tabular baseline for the current dataset
- These models work well when the relationship is mostly pointwise and the dataset is structured tabular data

### Physics-Informed LSTM
- PI-LSTM adds temporal memory through sequential modeling
- PI-LSTM adds physics-aware behavior through the custom loss design
- PI-LSTM is valuable as the advanced comparison model for hot outlet prediction
- On the current saved artifacts, it should be presented as a research extension, not as the confirmed overall best model

### Honest Takeaway For Version 1
1. Traditional ML gives the strongest current baseline on this dataset
2. PI-LSTM shows the next-stage modeling direction for sequential thermal systems
3. The dashboard can now compare baseline ML, hybrid adjustment, and PI-LSTM in one place
4. Hot outlet temperature is the main trustworthy target for PI-LSTM presentation

### Next Step
The strongest follow-up experiment is Version 2: train the models on carefully reduced data while preserving the full temperature range, then compare how traditional ML and PI-LSTM degrade under data scarcity.

## Step 22: Export Research Outputs

In [ ]:
primary_results["metrics_df"].to_csv("primary_target_metrics.csv", index=False)
primary_results["predictions_df"].to_csv("primary_target_predictions.csv", index=False)
primary_results["tuning_df"].to_csv("primary_target_tuning_log.csv", index=False)
secondary_results["metrics_df"].to_csv("secondary_target_metrics.csv", index=False)
secondary_results["predictions_df"].to_csv("secondary_target_predictions.csv", index=False)
secondary_results["tuning_df"].to_csv("secondary_target_tuning_log.csv", index=False)
with open("primary_target_artifact.pkl", "wb") as f:
    pickle.dump(primary_results["artifact"], f)
with open("secondary_target_artifact.pkl", "wb") as f:
    pickle.dump(secondary_results["artifact"], f)
print("Export complete.")


## Step 23: Download Files In Colab

In [ ]:
export_files = [
    "primary_target_metrics.csv",
    "primary_target_predictions.csv",
    "primary_target_tuning_log.csv",
    "secondary_target_metrics.csv",
    "secondary_target_predictions.csv",
    "secondary_target_tuning_log.csv",
    "primary_target_artifact.pkl",
    "secondary_target_artifact.pkl",
]
if files is not None:
    for file_name in export_files:
        files.download(file_name)
else:
    print("Download helper available only in Google Colab.")
    print("Generated files:", export_files)


## Appendix: Further Extension Ideas

This notebook already covers physics-aware feature engineering, leakage-safe training, from-scratch traditional ML baselines, and a PI-LSTM comparison section.

### Good next extensions
1. Add geometry-aware heat-transfer features such as area, diameter, velocity, Reynolds number, and Nusselt number
2. Add k-fold cross-validation
3. Add uncertainty intervals via bootstrapping
4. Build the separate low-data Version 2 study to test when LSTM / PI-LSTM becomes more useful
5. Compare from-scratch models against optimized library baselines as a validation benchmark